In [ ]:
import numpy as np
from magtense.magstatics import Tiles, run_simulation, get_demag_tensor, get_H_field_fmm, get_H_field

np.random.seed(42)

In [ ]:
# 1. Physical Constants
mu0 = 4 * np.pi * 1e-7
M_val = 1.2 / mu0  # ~954,930 A/m (Equivalent to 1.2 Tesla)

# 2. Geometry Setup (240nm bounding box)
LLC = np.array([0, 0, 0], dtype=np.float64)
URC = np.array([1, 1, 1], dtype=np.float64) * 240e-9
res = 20  
total_n = res**3

box_dim = URC - LLC
dx, dy, dz = box_dim / res

In [ ]:
# 3. Generate Centroids (Grid Positions)
x_c = np.linspace(LLC[0] + dx/2, URC[0] - dx/2, res)
y_c = np.linspace(LLC[1] + dy/2, URC[1] - dy/2, res)
z_c = np.linspace(LLC[2] + dz/2, URC[2] - dz/2, res)

X, Y, Z = np.meshgrid(x_c, y_c, z_c, indexing='ij')
offsets = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

# 4. Generate Easy Axis Directions (5-degree random cone)
max_angle_rad = np.radians(5.0)

phi = np.random.uniform(0, 2 * np.pi, total_n)
# Uniform sampling on a spherical cap
cos_theta = np.random.uniform(np.cos(max_angle_rad), 1.0, total_n)
sin_theta = np.sqrt(1 - cos_theta**2)

# Unit vectors for the Easy Axis (u_ea)
ux = sin_theta * np.cos(phi)
uy = sin_theta * np.sin(phi)
uz = cos_theta
u_ea_random = np.stack([ux, uy, uz], axis=1)

In [ ]:
# 5. Initialize MagTense Tiles
# We pass M_rem as a scalar; it applies to all 'total_n' tiles
tiles = Tiles(
    n=total_n, 
    M_rem=M_val, 
    tile_type=[2] * total_n
)

# Set the grid geometry
tiles.offset = offsets
tiles.size = np.tile([dx, dy, dz], (total_n, 1))

# Set the easy axis (This triggers M = M_rem * u_ea internally)
tiles.u_ea = u_ea_random
print(f"Setup complete: {total_n} tiles.")
print(f"M_rem: {M_val:.2e} A/m")
print(f"First tile Magnetization Vector:\n{tiles.M[0]}")

In [ ]:
pts = np.array( [ [240e-9, 240e-9, 240e-9]] ) * 5

In [ ]:
#H_fmm = get_H_field_fmm(tiles, offsets, eps=1e-4, nterms_in=10, cells_per_node=10, nlmin=0, nlmax=2, ifunif=1, do_target=0, do_FI=1, n_pts=total_n)
H_fmm = get_H_field_fmm(tiles, pts, eps=1e-4, nterms_in=10, cells_per_node=10, nlmin=0, nlmax=2, ifunif=1, do_target=1, do_FI=1, n_pts=1)

In [ ]:
H_fmm

In [ ]:
#H_mag = get_H_field(tiles, offsets)
H_mag = get_H_field(tiles, pts)

In [ ]:
H_mag

In [ ]:
H_fmm_old = get_H_field_fmm(tiles, offsets, eps=1e-4, nterms_in=10, cells_per_node=10, nlmin=0, nlmax=2, ifunif=1, do_target=0, do_FI=1, n_pts=total_n)
H_fmm_old

In [ ]:
H_fmm_new = get_H_field_fmm(tiles, offsets, eps=1e-4, nterms_in=17, cells_per_node=10, nlmin=0, nlmax=2, ifunif=0, do_target=0, do_FI=0, n_pts=total_n)
H_fmm_new

In [ ]:
np.max(np.abs(H_fmm_new - H_fmm_old))